# 04 – Climate Tourism Dashboard

Uses NASA POWER API to download daily climate variables for a selected tourism location.

In [ ]:
from pathlib import Path
import pandas as pd
import requests, json, os, re
ROOT = Path.cwd()
if ROOT.name == 'starter_notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
RAW = DATA / 'raw'
SAMPLE = DATA / 'sample'
RAW.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)

In [ ]:
import matplotlib.pyplot as plt

def get_nasa_power_daily(latitude, longitude, start='20240101', end='20241231'):
    params = 'T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN'
    url = f'https://power.larc.nasa.gov/api/temporal/daily/point?parameters={params}&community=RE&longitude={longitude}&latitude={latitude}&start={start}&end={end}&format=JSON'
    data = requests.get(url, timeout=60).json()
    p = data['properties']['parameter']
    dates = sorted(next(iter(p.values())).keys())
    rows=[]
    for d in dates:
        row={'date':pd.to_datetime(d)}
        for k,v in p.items(): row[k]=v.get(d)
        rows.append(row)
    return pd.DataFrame(rows)

# Luanda example
climate = get_nasa_power_daily(-8.8383, 13.2344)
climate.head()

In [ ]:
climate['month'] = climate['date'].dt.month
monthly = climate.groupby('month')[['T2M','PRECTOTCORR','RH2M','WS2M','ALLSKY_SFC_SW_DWN']].mean().reset_index()
monthly

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(monthly['month'], monthly['T2M'], marker='o')
plt.title('Average Monthly Temperature - Luanda Example')
plt.xlabel('Month')
plt.ylabel('Temperature at 2m (°C)')
plt.grid(True)
plt.show()

In [ ]:
# Simple tourism climate comfort score example
# This is only a prototype formula. Teams should explain assumptions.
monthly['comfort_score'] = 100 - abs(monthly['T2M'] - 24)*3 - monthly['PRECTOTCORR']*2
monthly[['month','T2M','PRECTOTCORR','comfort_score']].sort_values('comfort_score', ascending=False)

## How to extend

Run the same API for several tourism sites, compare best travel months, and combine climate data with attraction maps.